# Part 2: from a PostGIS database to an OGC API

In part 1 you published a GeoPackage. Real services at Kartverket follow the **golden path** instead ([docs](https://kartverket.github.io/ogcapi-docs/docs/ogcapi-skip)):

```
raw data ──▶ data platform (gold product) ──▶ PostGIS ──▶ pygeoapi + frontend
```

In this part the data already sits in a PostGIS database, the way a finished data product would. Your job is to publish it. Along the way you'll get three things a GeoPackage didn't give you in part 1:

- **Vector tiles** (OGC API – Tiles), cut on the fly from the database
- **Downloads** in the frontend: whole collections, or filtered by county or municipality
- **Advanced queries**: CQL2 filters, sorting and choosing attributes

The database runs here in your Codespace, so there's nothing to set up and nothing you can break. It's loaded from `postgis/adm.sql.gz` the first time it starts.

Run the cells from top to bottom (or **Run All**). The cells you're meant to edit are marked ✏️.

> If VS Code asks you to pick a kernel, choose **Python Environments → .venv** (marked *Recommended*).

In [ ]:
import json
from workshop import *

## 1. Start the database and download the software

The first time, this loads the workshop database and downloads Kartverket's images. It takes a couple of minutes, so feel free to read ahead.

In [ ]:
start_database()
download()

## 2. What's in the database? ✏️

The workshop's data lives in the schema `adm`, with one table per dataset. A well-kept database documents itself: every table and column has a **comment**. They're a good starting point for your metadata in steps 3 and 4.

**Want to use your own data?** Bring a *dump* of it: a file with the tables and their contents (see below for how to make one). Drag it into the `postgis/` folder in the Explorer on the left, remove the `#` in front of `load_dump`, fix the file name, and set `SCHEMA` to the schema your tables are in. Then adjust steps 3 and 4 to your tables.

In [ ]:
# load_dump("postgis/my-dump.sql.gz")

SCHEMA = "adm"
tables = inspect_postgis(SCHEMA)
describe_postgis(tables)

<details>
<summary><b>How to make a dump of your own data</b></summary>

`load_dump` accepts the common formats: plain SQL (`.sql`), gzipped SQL (`.sql.gz`), a `.zip` with one `.sql` file in it, and `pg_dump`'s custom format (`.dump`).

**From a PostGIS database**, with `pg_dump`, e.g. one schema:

```
pg_dump --no-owner --no-privileges --schema=my_schema -Fc -f my-dump.dump "postgresql://user@host:5432/database"
```

Use `--table=my_schema.my_table` instead of `--schema` for a single table. `--no-owner --no-privileges` leaves out the database's users and permissions, which don't exist here. For a Cloud SQL database, run `pg_dump` through the Cloud SQL Auth Proxy.

**From files** (GeoPackage, Shapefile, GeoJSON, …), with GDAL's `ogr2ogr`, without needing a database at all:

```
ogr2ogr -f PGDump my-dump.sql my-data.gpkg -lco SCHEMA=my_schema
```

That's how Kartverket's own development dumps are made.

**What pygeoapi needs from each table you publish:** a primary key (or another column with unique values), and a geometry column with a CRS that has an EPSG code.

Keep your dump in your Codespace: `postgis/` is set up so that git ignores everything in it apart from the workshop's own `adm.sql.gz`, so your data isn't committed by accident.
</details>

Curious? You can query the database directly, with ordinary SQL:

In [ ]:
for row in psql_json("""
    SELECT navn, fylke, areal_km2
    FROM adm.kommuner
    ORDER BY areal_km2 DESC
    LIMIT 5
"""):
    print(row)

## 3. Describe the dataset ✏️

The metadata for the service as a whole, as in part 1.

In [ ]:
DATASET = {
    "title": "Administrative enheter",
    "description": (
        "Norges fylker og kommuner fra Kartverkets administrative enheter, og et utvalg "
        "norske byer, publisert fra PostGIS. Laget for OGC API-workshopen, og ikke en "
        "autoritativ kilde: bruk Geonorge for de offisielle datasettene."
    ),
    "keywords": ["administrative enheter", "fylke", "kommune", "by", "workshop"],
    "provider": {
        "name": "Kartverket",
        "url": "https://www.kartverket.no",
    },
    "contact": {
        "name": "Kartverket",
        "email": "post@kartverket.no",
        "url": "https://www.kartverket.no",
    },
    "license": {
        "name": "CC BY 4.0",
        "url": "https://creativecommons.org/licenses/by/4.0/",
    },
}

## 4. Choose and describe the collections ✏️

**Only the tables you list here are published.** A real database usually holds more than you want to expose, so publishing is something you opt into, table by table. Keys are `schema.table`, as listed in step 2.

The fields are the same as in part 1 (`id`, `title`, `description`, `keywords`, `id_field`, `title_field`, `links`), plus one more:

- `download` (optional): lets people download a collection **filtered by county or municipality** in the frontend, e.g. "the towns in Vestland". Each filter names the collection to filter by, the column with the names to pick from, and a label. Kartverket's download process only knows two kinds of area, `fylke` and `kommune`, and expects those collections to be called `fylker` and `kommuner`.

In [ ]:
COLLECTIONS = {
    "adm.fylker": {
        "id": "fylker",
        "title": "Fylker",
        "description": "Norges fylker med fylkesnummer, navn, antall kommuner og areal.",
        "keywords": ["fylke", "fylkesgrense", "administrativ inndeling"],
        "id_field": "fylkesnummer",
        "title_field": "navn",
        "links": [
            {
                "title": "Administrative enheter – fylker i Geonorge",
                "href": "https://kartkatalog.geonorge.no/metadata/administrative-enheter-fylker/6093c8a8-fa80-11e6-bc64-92361f002671",
            },
        ],
    },
    "adm.kommuner": {
        "id": "kommuner",
        "title": "Kommuner",
        "description": "Norges kommuner med kommunenummer, navn, fylke og areal.",
        "keywords": ["kommune", "kommunegrense", "administrativ inndeling"],
        "id_field": "kommunenummer",
        "title_field": "navn",
        "links": [
            {
                "title": "Administrative enheter – kommuner i Geonorge",
                "href": "https://kartkatalog.geonorge.no/metadata/administrative-enheter-kommuner/041f1e6e-bdbc-4091-b48f-8a5990f3cc5b",
            },
        ],
        "download": {
            "area_filters": [
                {"type": "fylke", "collection": "fylker", "name_field": "navn", "label": "Fylke"},
            ],
        },
    },
    "adm.byer": {
        "id": "byer",
        "title": "Byer",
        "description": "Et utvalg på 25 norske byer, med kommunen og fylket de ligger i.",
        "keywords": ["by", "tettsted", "kommune", "fylke"],
        "id_field": "id",
        "title_field": "navn",
        "download": {
            "area_filters": [
                {"type": "fylke", "collection": "fylker", "name_field": "navn", "label": "Fylke"},
                {"type": "kommune", "collection": "kommuner", "name_field": "navn", "label": "Kommune"},
            ],
        },
    },
}

## 5. Generate the configuration

Compare the result with part 1. Each collection now has:

- a `PostgreSQL` provider, pointing at its table, with the schema in `search_path`. The connection details are `${DB_HOST}` and friends: placeholders filled in from the backend's environment, so no password ends up in the config. That's exactly how it works on SKIP.
- an `MVT-postgresql` provider: the vector tiles.

At the top there are also Kartverket's **processes**, which power the downloads.

In [ ]:
published = [t for t in tables if t["layer"] in COLLECTIONS]

config = build_config(DATASET, COLLECTIONS, published, "postgis")
write_config(config)

print(CONFIG_FILE.read_text(encoding="utf-8"))

## 6. Start your API 🚀

In [ ]:
start()
show_links(published)

Open the **landing page** link above. On a collection's page you'll now also find **tiles** and a **download** panel: try downloading the towns in one county as a GeoPackage.

## 7. Advanced queries

Everything below is plain OGC API, so it works the same from QGIS, a script or a browser.

In [ ]:
# Sorting, and choosing which attributes you get back:
# the five largest municipalities, without geometry
items = get_json("/collections/kommuner/items", {
    "sortby": "-areal_km2",
    "properties": "navn,fylke,areal_km2",
    "skipGeometry": "true",
    "limit": 5,
})
for f in items["features"]:
    print(f["id"], f["properties"])

In [ ]:
# CQL2 filters: conditions on several attributes at once
items = get_json("/collections/kommuner/items", {
    "filter": "fylke = 'Innlandet' AND areal_km2 > 2000",
    "filter-lang": "cql2-text",
    "properties": "navn,areal_km2",
    "skipGeometry": "true",
})
print(len(items["features"]), "large municipalities in Innlandet:")
for f in items["features"]:
    print(" ", f["properties"]["navn"], f["properties"]["areal_km2"], "km²")

In [ ]:
# Vector tiles: how a map client finds them. The API describes its tile sets
# (one per tiling scheme) and gives a URL template for fetching single tiles.
tiles = get_json("/collections/kommuner/tiles")
print("Tiling schemes:", [t["tileMatrixSetURI"].split("/")[-1] for t in tiles.get("tilesets", [])])
for link in tiles.get("links", []):
    if link.get("rel") == "item":
        print("Tile URL template:", link["href"])

### Processes

The downloads in the frontend are **processes** (OGC API – Processes): operations the server runs for you, with inputs you send along. You can list them and run them from code too.

A process is run with a **POST** to `/processes/<id>/execution`, with the inputs as JSON. That's why opening an `…/execution` link in the browser only gives *Method Not Allowed*: a browser sends a GET.

In [ ]:
# Which processes are there?
for p in get_json("/processes")["processes"]:
    print(f"{p['id']:24} {p['title']}")

In [ ]:
# Export the towns in Vestland as a GeoPackage: what the frontend's download panel does
gpkg = run_process("export-by-area-gpkg", {
    "collection_id": "byer",
    "area_type": "fylke",
    "area_name": "Vestland",
})
out = DATA_DIR / "byer_vestland.gpkg"
out.write_bytes(gpkg)
print(f"Saved {out.relative_to(ROOT)} ({len(gpkg) / 1024:.0f} KB). What's in it:\n")
describe(inspect_gpkg(out))

In [ ]:
# All the values of an attribute: what the frontend uses for its filter lists
result = run_process("get-distinct-values", {"resource_name": "kommuner", "queryable": "fylke"})
print(result.get("value", result) if isinstance(result, dict) else result)

**Try it yourself:** some more things to try in the cells above:

- `"filter": "navn LIKE 'Ve%'"`: names starting with "Ve"
- `"filter": "kommunenummer IN ('0301', '4601', '5001')"`: several municipalities by number (Oslo, Bergen, Trondheim)
- `/collections/kommuner/queryables`: everything you can filter on

## 8. Style the tiles yourself in QGIS (optional)

Why are the tiles green in the frontend? Because **vector tiles have no colours**. A tile only holds data: geometries cut to the tile, their attributes, and a layer name. How it looks is decided entirely by whatever draws it. Kartverket's frontend draws every collection in the same green. QGIS, MapLibre or Norgeskart can draw the very same tiles any way they like.

Try it: style the municipalities by county in QGIS.

1. Open the **Ports** tab at the bottom of VS Code, right-click port **3000** → **Port Visibility → Public**. Anyone with the link can now reach your API, so only do this with data you're happy to share.
2. Run the cell below to get the tile URLs in the form QGIS wants.
3. In QGIS: **Layer → Add Layer → Add Vector Tile Layer… → New → New Generic Connection**. Give it a name, paste the `kommuner` URL, set **Min. zoom** to 0 and **Max. zoom** to 18, then **OK** and **Add**.
4. Open the layer's **Properties → Symbology**. Each rule there styles one layer *inside* the tiles (`kommuner`). Double-click the polygon rule's symbol, click the **ε** button next to **Fill color** → **Edit…**, and enter:

   ```
   color_hsv(to_int("fylkesnummer") * 37 % 360, 55, 85)
   ```

   Every county now gets its own colour, straight from the tiles' `fylkesnummer` attribute.
5. Optional: under **Labels**, add a label for the `kommuner` layer with the expression `"navn"`.

Same API, same tiles, completely different map. That's also why there's a separate standard, OGC API – Styles, for publishing a recommended look *next to* the data.

In [ ]:
# Tile URLs for QGIS: it expects {z}/{x}/{y} placeholders, while the API's own
# template uses {tileMatrix}/{tileRow}/{tileCol} — that's z, then y (row), then x (column).
for t in published:
    print(f"{t['id']:10} {public_url(3000)}/collections/{t['id']}/tiles/WebMercatorQuad/{{z}}/{{y}}/{{x}}?f=mvt")

## 9. How this works on SKIP

The config you generated in step 5 is the one you'd deploy. In production the database is Cloud SQL instead of the container here, and the only thing that changes is where the `${DB_*}` placeholders get their values: `DB_HOST`, `DB_NAME`, `DB_USER` and `DB_PASSWORD` are set in the application's environment on SKIP, with the password kept as a secret. See [OGC API på SKIP](https://kartverket.github.io/ogcapi-docs/docs/ogcapi-skip) and the [starter kit](https://github.com/kartverket/ogc-api-starter) for the full setup.

## Stop

Stops the API and the database. The database keeps its data for next time.

To stop, remove the `#` and run the cell. It's commented out so that **Run All** doesn't stop your API straight after starting it.

In [ ]:
# stop()